# Spatio-temporal Hawkes processes

Events now carry a location as well as a time. The intensity is separable,

$$
\lambda(t, x \mid H_t) = \mu(x) + \sum_{t_i < t} \kappa_t(t - t_i)\,\kappa_s\!\left(d(x, x_i)\right),
$$

with $d$ the geodesic distance **on the domain** — so the domain is what decides
what "nearby" means, and everything else is written once.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import hawkes_package as hp

## The two kernels

`temporal` is a function of the elapsed time; `spatial` is a function of the
**distance** between two points of the domain, never of a signed offset. That is
what lets the same kernel be handed to a circle, a torus or a Klein bottle
unchanged.

In [ ]:
def temporal(lag):
    """A triangular kernel: excitation ramps up, peaks at 1, then decays away."""
    a, b = 0.9, 2.0
    rising = (lag > 0) & (lag < b / 2)
    falling = (lag >= b / 2) & (lag < b)
    return 2 * a / b * lag * rising + (-2 * a / b * lag + 2 * a) * falling


def spatial(distance):
    """A bump that decays to zero at distance pi/2 and stays there."""
    reach = np.pi / 2
    inside = np.abs(distance) <= reach
    return np.cos(np.pi * np.asarray(distance) / (2 * reach)) ** 2 * inside


lag = np.linspace(0, 2.5, 200)
gap = np.linspace(0, np.pi, 200)

fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3))
left.plot(lag, temporal(lag), "m")
left.set_xlabel("time since the event")
left.set_ylabel(r"$\kappa_t$")
right.plot(gap, spatial(gap), "r")
right.set_xlabel("distance from the event")
right.set_ylabel(r"$\kappa_s$")
fig.tight_layout()
plt.show()

## On a circle

`monotone_temporal_kernel=False` because this kernel rises before it decays: the
thinning bound then has to dominate the *peak* the kernel can still reach, not
merely its value now.

In [ ]:
# 25 events rather than 100: each one costs a quadrature sweep across space plus
# a Metropolis chain for the location, and this notebook runs on every docs build.
process = hp.SpatioTemporalHawkesProcess(
    base=lambda x: 0.5,
    spatial=spatial,
    temporal=temporal,
    domain=hp.Circle(),
    rng=42,
)
process.simulate(25)
print(f"{process.events.shape[1]} events, last at t = {process.events[0, -1]:.2f}")

## The intensity field

Straight from the process rather than re-derived here, which is what guarantees
the picture matches what the simulator actually thinned against.

In [ ]:
time, space, field = process.intensity_over_interval(
    np.linspace(0, process.events[0, -1], 200),
    points=np.linspace(-np.pi, np.pi, 200),
)
# `field` is (n_space, n_time): rows index space, columns index time.
field.shape

In [ ]:
plt.figure(figsize=(10, 6))
plt.contourf(time, space[:, 0], field, 50, cmap="jet")
plt.colorbar(label=r"$\lambda(t, x \mid H_t)$")
plt.scatter(process.events[0, :], process.events[1, :], c="r", marker="^", label="events")
plt.xlabel("time")
plt.ylabel("position on the circle")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Somewhere else entirely

`domain=` is the only thing that changes. Between them the built-in domains
reach every closed surface, and each reports which one it is.

In [ ]:
for domain in (hp.Circle(), hp.Torus2D(), hp.Sphere(), hp.FundamentalDomain.klein_bottle()):
    name = getattr(domain, "topology", None)
    print(f"{type(domain).__name__:20s} area {domain.volume:8.4f}  {name.name if name else ''}")

In [ ]:
# The Klein bottle: flat, like the torus, and non-orientable, unlike it. Eight
# events, because a two-dimensional domain costs about a thousand kernel
# evaluations per integration.
bottle = hp.FundamentalDomain.klein_bottle(2 * np.pi, 2 * np.pi)
on_bottle = hp.SpatioTemporalHawkesProcess(
    base=lambda x: 0.05,
    spatial=spatial,
    temporal=temporal,
    domain=bottle,
    rng=3,
)
on_bottle.simulate(8)

plt.figure(figsize=(5, 5))
plt.scatter(
    on_bottle.events[1, :],
    on_bottle.events[2, :],
    c=on_bottle.events[0, :],
    cmap="viridis",
)
plt.colorbar(label="time")
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"{on_bottle.events.shape[1]} events on the {bottle.topology.name}")
plt.tight_layout()
plt.show()